In [88]:
import copy
import torch
import math
import torch.nn as nn
import torch.optim
import torch.nn.parameter as Parameter
from enum import Enum, auto
from typing import Optional

# Reference paper: Attention is All You Need! https://arxiv.org/abs/1706.03762

# Implement the block and attention functions
# Bonus: multihead + autoregressive


In [89]:
class GPT2Config(object):
    class AttentionType(Enum):
        FlashAttention = auto()
        RegularAttention = auto()
    
    def __init__(
            self,
            vocab_size_or_config_json_file=50257,
            seq_len=32,
            n_embd=768,
            n_layer=6,
            n_head=4,
            layer_norm_epsilon=1e-5,
            initializer_range=0.02,
            attention=AttentionType.FlashAttention,
            q_tile_size = 4,
            kv_tile_size = 4,
    ):
        self.vocab_size = vocab_size_or_config_json_file
        self.seq_len = seq_len
        self.n_embd = n_embd
        self.n_layer = n_layer
        self.n_head = n_head
        self.layer_norm_epsilon = layer_norm_epsilon
        self.initializer_range = initializer_range
        self.attention = attention
        self.q_tile_size = q_tile_size
        self.kv_tile_size = kv_tile_size
        self.autoregressive_prefill = False
        self.chunked_prefill = False
        self.chunk_size = seq_len // 4

def gelu(x):
    return 0.5 * x * (1 + torch.tanh(math.sqrt(2 / math.pi) * (x + 0.044715 * torch.pow(x, 3))))

class LayerNorm(nn.Module):
    def __init__(self, hidden_size, eps=1e-12):
        super(LayerNorm, self).__init__()
        self.weight = nn.Parameter(torch.ones(hidden_size))
        self.bias = nn.Parameter(torch.zeros(hidden_size))
        self.variance_epsilon = eps

    def forward(self, x):
        u = x.mean(-1, keepdim=True)
        s = (x - u).pow(2).mean(-1, keepdim=True)
        x = (x - u) / torch.sqrt(s + self.variance_epsilon)
        return self.weight * x + self.bias

class MLP(nn.Module):
    def __init__(self, dim, mid_layer_mult=4):
        super(MLP, self).__init__()
        self.linear1 = nn.Linear(dim, dim*mid_layer_mult)
        self.linear2 = nn.Linear(dim*mid_layer_mult, dim)
        self.act = gelu

    def forward(self, x):
        h = self.act(self.linear1(x))
        return self.linear2(h)


In [ ]:
class Block(nn.Module):
  def __init__(self, seq_len, config):
      super(Block, self).__init__()
      embed_size = config.n_embd
      self.attn_type = config.attention
      self.max_seq_len = config.seq_len * 4
      self.max_batch_size = 4
      self.register_buffer("kv_cache", 
                           torch.zeros(
                               self.max_batch_size, 
                               config.n_head, 
                               self.max_seq_len, 
                               (embed_size * 2) // config.n_head), 
                            persistent=False)
      self.ln_1 = LayerNorm(embed_size, eps=config.layer_norm_epsilon)
      self.attn = AttentionBlock(embed_size, seq_len, config, self.kv_cache)
      self.ln_2 = LayerNorm(embed_size, eps=config.layer_norm_epsilon)
      self.mlp = MLP(dim=embed_size)

  def forward(self, x, decode: bool = False, pos: int = 0): # x: [batch, seq_length, embed_dim]
      # for every layer: -> x -> attn -> ln1 -> MLP -> ln2 -> x
      self.attn.kv_cache = self.kv_cache
      x1 = self.attn(x, decode=decode, pos=pos)
      x2 = self.ln_1(x + x1)
      x3 = self.mlp(x2)
      x = self.ln_2(x3 + x2)
      return x

class AttentionBlock(nn.Module):
    class FullAttention(nn.Module):
        def __init__(self, embed_size, seq_len, config):
            super().__init__()
            assert embed_size % config.n_head == 0
            self.max_seq_len = seq_len * 4
            self.register_buffer("bias", torch.tril(torch.ones(self.max_seq_len, self.max_seq_len)).view(1, 1, self.max_seq_len, self.max_seq_len))
            self.n_head = config.n_head
            self.seq_len = seq_len
        
        def forward(self, x_q, x_k, x_v, start_pos : int = 0) -> torch.Tensor: # returns x [B, T, D]
            B, _, T, D_head = x_q.shape # [B, n_head, T, D // n_head]
            K_len = x_k.shape[-2]
            assert x_v.shape[-2] == K_len
            assert self.n_head == x_q.shape[1]
            D = D_head * self.n_head
            assert T <= K_len
            assert start_pos == K_len - T
            kv_len = K_len
            # Assume T=4, K_len=64, qtile0: x_q[0:4] and x_k[0:4]; qtile1: x_q[4:8] and x_k[0:8] ...
            # x_k and x_k have already been contracted causally so no updates
            scores = torch.matmul(x_q, x_k.transpose(-2, -1)) # [B, nHead, T, D_h] .x. [B, nHead, D_h, kv_len]
            scores = scores / math.sqrt(D_head) # = [B, nHead, T, kv_len]
            causal_mask = self.bias[:, :, start_pos:kv_len, :kv_len]
            scores = scores.masked_fill(causal_mask == 0, float("-inf"))
            probs = torch.softmax(scores, dim=-1)
            x = torch.matmul(probs, x_v) # [B, nHead, T, kv_len] .x. [B, nHead, kv_len, D_h] = [B, nHead, T, D_h]
            x = x.permute(0, 2, 1, 3) # [B, T, nHead, D_h]
            x = x.reshape(B, T, D)
            return x

    class FlashAttention(nn.Module):
        def __init__(self, embed_size, seq_len, config):
            super().__init__()
            assert embed_size % config.n_head == 0
            self.max_seq_len = seq_len * 4
            self.register_buffer("bias", torch.tril(torch.ones(self.max_seq_len, self.max_seq_len)).view(1, 1, self.max_seq_len, self.max_seq_len))
            self.n_head = config.n_head
            self.seq_len = seq_len
            self.q_tile_size = config.q_tile_size
            self.kv_tile_size = config.kv_tile_size
            assert self.q_tile_size == self.kv_tile_size
        
        def forward(self, x_q, x_k, x_v, start_pos : int = 0) -> torch.Tensor: # returns x [B, T, D]
            B, _, T, D_head = x_q.shape # [B, n_head, T, D // n_head]
            K_len = x_k.shape[-2]
            assert x_v.shape[-2] == K_len
            assert self.n_head == x_q.shape[1]
            D = D_head * self.n_head
            assert T <= K_len
            assert start_pos == K_len - T
            kv_len = K_len
            # Assume T=4, K_len=64, qtile0: x_q[0:4] and x_k[0:4]; qtile1: x_q[4:8] and x_k[0:8] ...
            # x_k and x_k have already been contracted causally so no updates
            num_q_tiles = math.ceil(T / self.q_tile_size)
            num_kv_tiles = math.ceil(kv_len / self.kv_tile_size)
            x = x_q.new_zeros(B, self.n_head, T, D_head)
            for q_tile in range(num_q_tiles):
                q_tile_start = q_tile * self.q_tile_size
                q_tile_end = min(T, (q_tile + 1) * self.q_tile_size)
                q_tile_len = q_tile_end - q_tile_start
                q_abs_start = start_pos + q_tile_start
                q_abs_end = start_pos + q_tile_end
                x_q_chunk = x_q[:, :, q_tile_start:q_tile_end, :] # [B, nHead, <=QTileSize, D_h]
                if q_tile_len < self.q_tile_size:
                    q_pad = x_q.new_zeros(B, self.n_head, self.q_tile_size - q_tile_len, D_head)
                    x_q_chunk = torch.cat((x_q_chunk, q_pad), dim=-2)
                q_valid = torch.arange(self.q_tile_size, device=x_q.device).view(1, 1, self.q_tile_size, 1) < q_tile_len
                max_old = x_q.new_full((B, self.n_head, self.q_tile_size, 1), float("-inf"))
                max_old = max_old.masked_fill(~q_valid, 0.0)
                sum_old = x_q.new_zeros(B, self.n_head, self.q_tile_size, 1)
                sum_old = sum_old.masked_fill(~q_valid, 1.0)
                x_out = x_q.new_zeros(B, self.n_head, self.q_tile_size, D_head)
                for kv_tile in range(num_kv_tiles):
                    kv_tile_start = kv_tile * self.kv_tile_size
                    kv_tile_end = min(kv_len, (kv_tile + 1) * self.kv_tile_size)
                    kv_tile_len = kv_tile_end - kv_tile_start
                    if kv_tile_start >= q_abs_end:
                        continue
                    x_k_chunk = x_k[:, :, kv_tile_start:kv_tile_end, :] # [B, nHead, <=KVTileSize, D_h]
                    x_v_chunk = x_v[:, :, kv_tile_start:kv_tile_end, :] # [B, nHead, <=KVTileSize, D_h]
                    if kv_tile_len < self.kv_tile_size:
                        kv_pad = x_k.new_zeros(B, self.n_head, self.kv_tile_size - kv_tile_len, D_head)
                        x_k_chunk = torch.cat((x_k_chunk, kv_pad), dim=-2)
                        x_v_chunk = torch.cat((x_v_chunk, kv_pad), dim=-2)
                    scores = torch.matmul(x_q_chunk, x_k_chunk.transpose(-2,-1)) # [B, nHead, QTileSize, KVTileSize]
                    scores /= math.sqrt(D_head) # [B, nHead, QTileSize, KVTileSize]
                    causal_mask = torch.zeros(1, 1, self.q_tile_size, self.kv_tile_size, dtype=torch.bool, device=x_q.device)
                    causal_mask[:, :, :q_tile_len, :kv_tile_len] = self.bias[:, :, q_abs_start:q_abs_end, kv_tile_start:kv_tile_end].bool()
                    scores = scores.masked_fill(~causal_mask, float("-inf")) # [B, nHead, QTileSize, KVTileSize]
                    max_new = scores.max(dim=-1, keepdim=True).values # [B, nHead, QTileSize, 1]
                    max_new = torch.maximum(max_old, max_new)
                    probs = torch.exp(scores - max_new) # [B, nHead, QTileSize, KVTileSize]
                    probs = probs.masked_fill(~causal_mask, 0.0)
                    sum_new = probs.sum(dim=-1, keepdim=True) # [B, nHead, QTileSize, 1]
                    sum_new = sum_new + sum_old * torch.exp(max_old - max_new)
                    x_out = x_out * torch.exp(max_old - max_new) * (sum_old / sum_new) + (torch.matmul(probs/sum_new, x_v_chunk))
                    # x_out = [B, nHead, QTileSize, D_h]
                    max_old = max_new
                    sum_old = sum_new
                x[:, :, q_tile_start:q_tile_end, :] = x_out[:, :, :q_tile_len, :]
                # x = [B, nHead, T, D_h]
            x = x.permute(0, 2, 1, 3) # [B, T, nHead, D_h]
            x = x.reshape(B, T, D)
            return x

    def __init__(self, embed_size, seq_len, config, kv_cache):
        super().__init__()
        self.n_head = config.n_head
        self.split_size = embed_size
        self.softmax = nn.Softmax(dim=-1)
        self.linear_qkv = nn.Linear(embed_size, embed_size * 3) # [B, T, D] -> [B, T, 3D] or [B, N, T, 3D/N]
        self.linear_out = nn.Linear(embed_size, embed_size)
        self.seq_len = seq_len
        self.max_seq_len = seq_len * 100
        self.kv_cache = kv_cache
        self.autoregressive_prefill = config.autoregressive_prefill
        self.chunked_prefill = config.chunked_prefill
        self.chunk_size = config.chunk_size
        self.attn = self.FullAttention(embed_size, seq_len, config)
        if (config.attention == config.AttentionType.FlashAttention):
            self.attn = self.FlashAttention(embed_size, seq_len, config)

    def forward(self, x, decode: bool = False, pos: int = 0): 
        # x: [batch, seq_length, embed_dim]
        # kv_cache: [max_batch, n_head, max_seq_len, 2 * head_dim]
        B, T, D = x.shape
        x_qkv = self.linear_qkv(x) # [B, T, 3 * D]
        x_qkv = x_qkv.reshape(B, T, self.n_head, (3 * D // self.n_head)) # [B, T, n_head, 3 * D // n_head]
        hidden_dim_size = D // self.n_head
        x_q = x_qkv[:, :, :, 0:hidden_dim_size] # [B, T, nHead, D // n_head]
        x_q = x_q.permute(0, 2, 1, 3) # [B, nHead, T, D // n_head]
        x_k = x_qkv[:, :, :, hidden_dim_size:2*hidden_dim_size] # [B, T, nHead, D // n_head]
        x_k = x_k.permute(0, 2, 1, 3) # [B, nHead, T, D // n_head]
        x_v = x_qkv[:, :, :, 2*hidden_dim_size:]
        x_v = x_v.permute(0, 2, 1, 3) # [B, nHead, T, D // n_head]
        assert B <= self.kv_cache.shape[0]
        assert T <= self.kv_cache.shape[2]
        # At this point we have
        # Prefill:
        #   x_q, x_v, x_k = [B, nHead, T, D_h]
        # Decode:
        #   x_q, x_v, x_k = [B, nHead, 1, D_h]
        if self.autoregressive_prefill and not decode:
            chunk_start_pos = 0
            num_chunks = T # 1 token per chunk for autoregressive
            x = x_q.new_zeros(B, T, D)
            for chunk_start_pos in range(num_chunks):
                x_q_chunk = x_q[:, :, chunk_start_pos:(chunk_start_pos + 1), :]
                x_k_chunk = x_k[:, :, :(chunk_start_pos + 1), :]
                x_v_chunk = x_v[:, :, :(chunk_start_pos + 1), :]
                x[:, chunk_start_pos:(chunk_start_pos + 1), :] = self.attn(x_q_chunk, x_k_chunk, x_v_chunk, chunk_start_pos)
            x = self.linear_out(x)
            return x
        elif self.chunked_prefill and not decode:
            num_chunks = math.ceil(T / self.chunk_size)
            x = x_q.new_zeros(B, T, D)
            for chunk_idx in range(num_chunks):
                chunk_start_pos = chunk_idx * self.chunk_size
                chunk_end_pos = min(T, chunk_start_pos + self.chunk_size)
                x_q_chunk = x_q[:, :, chunk_start_pos:chunk_end_pos, :]
                x_k_chunk = x_k[:, :, :chunk_end_pos, :]
                x_v_chunk = x_v[:, :, :chunk_end_pos, :]
                x[:, chunk_start_pos:chunk_end_pos, :] = self.attn(x_q_chunk, x_k_chunk, x_v_chunk, chunk_start_pos)
            x = self.linear_out(x)
            return x
        elif not decode: # full prefill
            # prefill
            self.kv_cache[:B, :, :T, :hidden_dim_size].copy_(x_k)
            self.kv_cache[:B, :, :T, hidden_dim_size:].copy_(x_v)
            x = self.attn(x_q, x_k, x_v)
            x = self.linear_out(x)
        else:
            assert T == 1 # 1-token only for decode
            # decode, update KV-cache
            assert pos + 1 <= self.kv_cache.shape[2]
            self.kv_cache[:B, :, pos:pos+1, :hidden_dim_size].copy_(x_k)
            self.kv_cache[:B, :, pos:pos+1, hidden_dim_size:].copy_(x_v)
            # get full KV for this decode step
            x_k = self.kv_cache[:B, :, :pos+1, :hidden_dim_size]
            x_v = self.kv_cache[:B, :, :pos+1, hidden_dim_size:]
            x = self.attn(x_q, x_k, x_v)
            x = self.linear_out(x)
        return x


In [91]:
class GPT2Model(nn.Module):
    def __init__(self, config):
        super(GPT2Model, self).__init__()
        self.n_layer = config.n_layer
        self.n_embd = config.n_embd
        self.n_vocab = config.vocab_size

        self.wte = nn.Embedding(config.vocab_size, config.n_embd)
        self.wpe = nn.Embedding(config.seq_len, config.n_embd)
        block = Block(config.seq_len, config)
        self.h = nn.ModuleList([copy.deepcopy(block) for _ in range(config.n_layer)])
        self.ln_f = LayerNorm(config.n_embd, eps=config.layer_norm_epsilon)

        embed_shape = self.wte.weight.shape # [V, D]
        self.decoder = nn.Linear(embed_shape[1], embed_shape[0], bias=False)

        self.loss_func = nn.CrossEntropyLoss(ignore_index=-1)


    def forward(self, input_ids, lm_labels):

        position_ids = torch.arange(0, input_ids.size(-1), dtype=torch.long,
                                    device=input_ids.device)
        position_ids = position_ids.unsqueeze(0).expand_as(input_ids)

        input_shape = input_ids.size()
        # below is a common technique to flatten batch dim
        input_ids = input_ids.view(-1, input_ids.size(-1))
        position_ids = position_ids.view(-1, position_ids.size(-1))
        inputs_embeds = self.wte(input_ids)
        position_embeds = self.wpe(position_ids)
        hidden_states = inputs_embeds + position_embeds

        ## For understanding
        # print(f"Input IDs -> {input_ids}")
        # print(f"Position IDs -> {position_ids}")
        # print(f"Input Embeds -> {inputs_embeds}")
        # print(f"Position Embeds -> {position_embeds}")
        # print(f"Hidden States -> {hidden_states}")

        for block in self.h:
            hidden_states = block(hidden_states)

        hidden_states = self.ln_f(hidden_states)
        output_shape = input_shape + (hidden_states.size(-1),)
        hidden_states = hidden_states.view(*output_shape)
        lm_logits = self.decoder(hidden_states)
        loss = self.loss_func(lm_logits.view(-1, lm_logits.size(-1)), lm_labels.view(-1))
        return loss


In [95]:
# Naive training loop - loss should go down

config = GPT2Config()
model = GPT2Model(config)

batch_size = 1
seq_len = 32
inputs = torch.range(1,seq_len,dtype=torch.int64)
inputs = torch.reshape(inputs,(batch_size,seq_len))

# Train loop
optimizer = torch.optim.SGD(model.parameters(), lr=0.001, momentum=0.9)
for i in range(20):
    # zero the parameter gradients
    optimizer.zero_grad()
    # forward + backward + optimize
    loss = model(input_ids=inputs[:,:-1],lm_labels=inputs[:,1:])
    loss.backward()
    optimizer.step()
    print("step ",i ," loss: ", loss.item())



/var/folders/4y/kyvkvxw15yx8ycxwq5q_nrth0000gn/T/ipykernel_64176/500661588.py:8: UserWarning: torch.range is deprecated and will be removed in a future release because its behavior is inconsistent with Python's range builtin. Instead, use torch.arange, which produces values in [start, end).
  inputs = torch.range(1,seq_len,dtype=torch.int64)


step  0  loss:  11.000323295593262
step  1  loss:  10.94830322265625
step  2  loss:  10.849477767944336
step  3  loss:  10.708616256713867
step  4  loss:  10.530220985412598
step  5  loss:  10.3186616897583
step  6  loss:  10.07828140258789
step  7  loss:  9.813446998596191
step  8  loss:  9.528509140014648
step  9  loss:  9.227702140808105
step  10  loss:  8.914994239807129
step  11  loss:  8.593889236450195
step  12  loss:  8.267277717590332
step  13  loss:  7.937336444854736
step  14  loss:  7.605503559112549
step  15  loss:  7.272549629211426
step  16  loss:  6.9387102127075195
step  17  loss:  6.603858470916748
step  18  loss:  6.267691135406494
step  19  loss:  5.929911136627197


In [ ]:
# PyTorch basics

a_ref = torch.zeros(4, 8, 128)
a = torch.rand_like(a_ref, dtype=torch.float16)
print(f"a.shape = {a.shape}, a.size = {a.size()}")
a = a.view(-1, a.size(-1))
print(f"a.shape = {a.shape}, a.size = {a.size()}")

a.shape = torch.Size([4, 8, 128]), a.size = torch.Size([4, 8, 128])
a.shape = torch.Size([32, 128]), a.size = torch.Size([32, 128])
